In [0]:
%sql
-- silver layer crm customer info delta table
create table silver_crm_cust_info
using delta 
as
select
  cst_id as customer_id,
  cst_key as customer_key,
  trim(cst_firstname) as firstname,
  trim(cst_lastname) as lastname,
  case trim(cst_marital_status)
    when 'S' then 'Single'
    when 'M' then 'Married'
    else 'N/A'
  end as marital_status,
  case trim(cst_gndr)
    when 'M' then 'Male'
    when 'F' then 'Female'
    else 'N/A'
  end as gender,
  cst_create_date as customer_creation_date
from (
	select *,
	row_number() over (partition by cst_id order by cst_create_date desc) as dup_count
	from bronze_crm_cust_info
	where cst_id is not null
) as t 
where dup_count = 1

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
%sql
-- silver layer crm product info delta table 
create table silver_crm_prd_info
using delta
as
select 
  prd_id,
  replace(substring(prd_key, 1, 5), '-', '_') as category_id, -- this is a derived column, extracting the category key of another table
  substring(prd_key, 7, len(prd_key)) as product_key, 
  prd_nm,
  coalesce(prd_cost, 0) as product_cost, -- price/cost cannot be null
  case upper(trim(prd_line))
    when 'R' then 'Road'
    when 'S' then 'Other Sales'
    when 'M' then 'Mountain'
    when 'T' then 'Touring'
    else 'N/A'
  end as product_line,
  prd_start_dt, -- databricks already inferred the schema. i don't need to cast this as date
  lead(prd_start_dt) over (partition by prd_key order by prd_start_dt) -1 as prd_end_dt

from workspace.`lakehouse-transformation-project`.bronze_crm_prd_info

In [0]:
%sql
-- silver layer crm sales details 
create table silver_crm_sales_details
using delta 
as
select
  sls_ord_num as order_number,
  sls_prd_key as product_key,
  sls_cust_id as customer_id,
  case 
    when sls_order_dt = 0 or length(cast(sls_order_dt as string)) != 8 then null
    else to_date(cast(sls_order_dt as string), 'yyyyMMdd')
  end as order_date,
  case 
    when sls_ship_dt = 0 or length(cast(sls_ship_dt as string)) != 8 then null
    else to_date(cast(sls_ship_dt as string), 'yyyyMMdd')
  end as ship_date,
  case 
    when sls_due_dt = 0 or length(sls_due_dt) != 8 then null
    else to_date(cast(sls_due_dt as string), 'yyyyMMdd')
  end as due_date,
  case 
    when sls_sales is null or sls_sales <= 0 or sls_sales != sls_quantity * abs(sls_price) 
    then sls_quantity * abs(sls_price)
    else sls_sales
  end as sales,
  sls_quantity as quantity,
  case 
    when sls_price is null or sls_price <= 0
    then sls_sales / nullif(sls_quantity,0)
    else sls_price
  end as price,
  current_timestamp() as dlh_create_date -- getdate() equivalent for when i uploaded these onto databricks
from workspace.`lakehouse-transformation-project`.bronze_crm_sales_details

In [0]:
%sql
create table silver_erp_cust_az_12
using delta
as
select 
  case 
    when CID like "NAS%" then substring(cid, 4, length(cid))
    else cid
  end as cid,
  case 
    when BDATE > current_date() then null
    else bdate
  end as birthdate,
  case 
    when (trim(GEN)) in ('F', 'Female') then 'Female'
    when (trim(GEN)) in ('M', 'Male') then 'Male'
    else 'N/A'
  end as gender,
  current_timestamp() as dlh_create_date 
from bronze_erp_cust_az_12

In [0]:
%sql
create table silver_erp_loc_a_101
using delta
as
select 
  replace(CID, '-', '') as cid,
  case 
    when trim(cntry) in ('USA', 'US', 'United States') then 'United States of America'
    when trim(cntry) in ('CAN', 'CA') then 'Canada'
    when trim(cntry) in ('DE', 'Germany') then 'Germany'
    when trim(cntry) in ('AU', 'Australia') then 'Australia'
    when trim(cntry) = '' or cntry is null then 'N/A'
    else trim(cntry)
  end as country
from bronze_erp_loc_a_101

In [0]:
%sql
create table silver_px_cat_g_1_v_2
using delta
as
select 
    ID,
    trim(CAT) as CAT,
    SUBCAT, 
    MAINTENANCE
from bronze_erp_px_cat_g_1_v_2